# 02. Preprocessing Policy

이 노트북은 raw 전처리 노트북이 아닙니다.

팀원 간 합의로 만들어진 v1 기준 데이터(`Membership_v1.csv`, `User_Mapping_v1.csv`, `View_History_v1.csv`, `Movie_Master_v1.csv`)를 입력으로 받아, 이번 100원딜 OTT 이탈 분석에 사용할 analysis base를 만듭니다.

핵심 원칙은 다음입니다.

- raw 데이터는 다시 전처리하지 않습니다.
- v1 데이터는 팀 합의 기준 데이터로 간주합니다.
- 이번 노트북에서는 분석 단위, 분석 제외 기준, 100원딜 flag, 3주 관측창만 고정합니다.
- 산출물은 다음 단계가 실제로 필요로 하는 파일과 최소 검산표만 저장합니다.

## 2-1. 경로 설정

현재 저장소 구조는 다음을 전제로 합니다.

```text
<repo>/
├─ _data/
│  ├─ 01_raw/
│  └─ 02_interim/
│     └─ 260430 membership_v1(상치, 이름 변경)/
│        ├─ Membership_v1.csv
│        ├─ User_Mapping_v1.csv
│        ├─ View_History_v1.csv
│        └─ Movie_Master_v1.csv
└─ park.ingyeom/
   ├─ notebooks/
   └─ reports/
```

노트북은 `park.ingyeom/notebooks`에서 실행되지만, 데이터는 `park.ingyeom/_data`가 아니라 저장소 루트의 `_data`를 사용합니다.

In [26]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


def find_project_root(start: Path | None = None) -> Path:
    """
    저장소 루트를 찾는다.

    주의:
    - 노트북은 <repo>/park.ingyeom/notebooks 아래에 있다.
    - 실제 공유 데이터는 <repo>/_data 아래에 있다.
    - 따라서 <repo>/park.ingyeom/_data 를 프로젝트 루트로 잡으면 안 된다.
    """
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / ".git").exists() and (candidate / "_data").exists():
            return candidate

    for candidate in candidates:
        interim_dir = candidate / "_data" / "02_interim"
        if interim_dir.exists() and any(interim_dir.rglob("Membership_v1.csv")):
            return candidate

    for candidate in candidates:
        if candidate.name == "park.ingyeom" and (candidate.parent / "_data").exists():
            return candidate.parent

    raise FileNotFoundError(
        "저장소 루트를 찾지 못했습니다. "
        "노트북은 <repo>/park.ingyeom/notebooks 안에서 실행하고, "
        "데이터는 <repo>/_data 아래에 있어야 합니다."
    )


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "_data"

WORK_ROOT = PROJECT_ROOT / "park.ingyeom"

REPORTS_DIR = WORK_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"

NOTEBOOK_ID = "02_preprocessing_policy"

OUTPUT_DATA_DIR = REPORTS_DIR / "data" / NOTEBOOK_ID
OUTPUT_TABLE_DIR = TABLES_DIR / NOTEBOOK_ID

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

V1_DATA_CANDIDATES = [
    DATA_ROOT / "02_interim" / "260430 membership_v1(상치, 이름 변경)",
    DATA_ROOT / "02_interim" / "260430 membership_v1(이상치, 이름 변경)",
    DATA_ROOT / "02_interim" / "260430_membership_v1(상치, 이름 변경)",
    DATA_ROOT / "02_interim" / "260430_membership_v1(이상치, 이름변경)",
    DATA_ROOT / "02_interim" / "260430_membership_v1(이상치, 이름 변경)",
]

# 명시 후보가 실패하면, 02_interim 바로 아래의 membership_v1 폴더만 보조 후보로 탐색한다.
if (DATA_ROOT / "02_interim").exists():
    V1_DATA_CANDIDATES.extend(
        sorted((DATA_ROOT / "02_interim").glob("*membership*v1*"))
    )

V1_DATA_DIR = next((p for p in V1_DATA_CANDIDATES if p.exists()), None)
if V1_DATA_DIR is None:
    raise FileNotFoundError(
        "팀 합의 기준 v1 데이터 폴더를 찾지 못했습니다. "
        "예상 위치 예시: <repo>/_data/02_interim/260430 membership_v1(상치, 이름 변경)/"
    )

PATH_MEMBERSHIP = V1_DATA_DIR / "Membership_v1.csv"
PATH_MAPPING = V1_DATA_DIR / "User_Mapping_v1.csv"
PATH_VIEW = V1_DATA_DIR / "View_History_v1.csv"
PATH_MOVIE_MASTER = V1_DATA_DIR / "Movie_Master_v1.csv"

REQUIRED_FILES = {
    "membership": PATH_MEMBERSHIP,
    "mapping": PATH_MAPPING,
    "view": PATH_VIEW,
    "movie_master": PATH_MOVIE_MASTER,
}

missing_files = {name: path for name, path in REQUIRED_FILES.items() if not path.exists()}
if missing_files:
    raise FileNotFoundError(
        "팀 합의 기준 v1 데이터 파일을 찾지 못했습니다.\n"
        + "\n".join([f"{name}: {path}" for name, path in missing_files.items()])
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("WORK_ROOT:", WORK_ROOT)
print("REPORTS_DIR:", REPORTS_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("V1_DATA_DIR:", V1_DATA_DIR)
print("OUTPUT_DATA_DIR:", OUTPUT_DATA_DIR)
print("OUTPUT_TABLE_DIR:", OUTPUT_TABLE_DIR)
print()

for name, path in REQUIRED_FILES.items():
    print(f"{name}: {path}")

PROJECT_ROOT: c:\Code\ott-churn-prediction
DATA_ROOT: c:\Code\ott-churn-prediction\_data
WORK_ROOT: c:\Code\ott-churn-prediction\park.ingyeom
REPORTS_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports
TABLES_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables
V1_DATA_DIR: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)
OUTPUT_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\02_preprocessing_policy
OUTPUT_TABLE_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables\02_preprocessing_policy

membership: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\Membership_v1.csv
mapping: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\User_Mapping_v1.csv
view: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\View_History_v1.csv
movie_master: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\Movie_Master_v1.csv


## 2-2. 이전 02번 산출물 정리

02번은 최소 산출물만 남깁니다.  
기존에 잘못 생성된 `02_*.csv`가 남아 있으면 현재 실행 결과와 섞이므로, 이 노트북은 실행 시작 시 기존 02번 report CSV와 02번 output data를 정리합니다.

삭제 대상은 다음으로 제한합니다.

- `park.ingyeom/reports/tables/02_*.csv`
- `_data/02_interim/park_ingyeom_outputs/02_preprocessing_policy/` 내부 파일

In [27]:
CLEAN_PREVIOUS_02_OUTPUTS = True

if CLEAN_PREVIOUS_02_OUTPUTS:
    removed_report_files = []
    for path in OUTPUT_TABLE_DIR.glob("02_*.csv"):
        path.unlink()
        removed_report_files.append(path.name)

    removed_output_files = []
    for path in OUTPUT_DATA_DIR.glob("*"):
        if path.is_file():
            path.unlink()
            removed_output_files.append(path.name)

    print("Removed previous 02 report files:", len(removed_report_files))
    if removed_report_files:
        print(removed_report_files)

    print("Removed previous 02 output files:", len(removed_output_files))
    if removed_output_files:
        print(removed_output_files)
else:
    print("Previous 02 outputs were not removed.")

Removed previous 02 report files: 7
['02_filter_summary.csv', '02_multi_usernum_membership_rows.csv', '02_observation_window_summary.csv', '02_repurchase_by_100won.csv', '02_repurchase_by_100won_maxscreen.csv', '02_user_mapping_summary.csv', '02_watch_presence_summary.csv']
Removed previous 02 output files: 4
['membership_preprocessed.csv', 'membership_with_usernum.csv', 'preprocessing_summary.json', 'view_history_observation_window.csv']


## 2-3. 데이터 로딩

여기서 읽는 데이터는 raw가 아니라 팀 합의 기준 v1 데이터입니다.

In [28]:
membership_base = pd.read_csv(PATH_MEMBERSHIP)
mapping_base = pd.read_csv(PATH_MAPPING)
view_base = pd.read_csv(PATH_VIEW)
movie_master_base = pd.read_csv(PATH_MOVIE_MASTER)

input_file_summary = pd.DataFrame([
    {"name": "membership_base", "path": str(PATH_MEMBERSHIP), "rows": len(membership_base), "cols": membership_base.shape[1]},
    {"name": "mapping_base", "path": str(PATH_MAPPING), "rows": len(mapping_base), "cols": mapping_base.shape[1]},
    {"name": "view_base", "path": str(PATH_VIEW), "rows": len(view_base), "cols": view_base.shape[1]},
    {"name": "movie_master_base", "path": str(PATH_MOVIE_MASTER), "rows": len(movie_master_base), "cols": movie_master_base.shape[1]},
])

display(input_file_summary)

,name,path,rows,cols
0,membership_base,c:\Code\ott-churn-prediction\_data\02_interim\...,17876,15
1,mapping_base,c:\Code\ott-churn-prediction\_data\02_interim\...,19877,2
2,view_base,c:\Code\ott-churn-prediction\_data\02_interim\...,106205,5
3,movie_master_base,c:\Code\ott-churn-prediction\_data\02_interim\...,14018,3


## 2-4. 기본 타입 정리

날짜 컬럼을 datetime으로 변환하고, 이번 분석용 `membership_row_id`를 부여합니다.  
`membership_row_id`는 개인 단위가 아니라 구독 이벤트 단위 식별자입니다.

In [29]:
membership = membership_base.copy()
mapping = mapping_base.copy()
view = view_base.copy()
movie_master = movie_master_base.copy()

membership["reg_date"] = pd.to_datetime(membership["reg_date"])
membership["end_date"] = pd.to_datetime(membership["end_date"])
watch_day_raw = (
    view["watch_day"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

view["watch_day"] = pd.to_datetime(
    watch_day_raw,
    format="%Y%m%d",
    errors="coerce",
)

membership = membership.reset_index(drop=True)
membership.insert(0, "membership_row_id", np.arange(len(membership), dtype=int))

membership["subscription_days"] = (membership["end_date"] - membership["reg_date"]).dt.days + 1

print("membership rows:", len(membership))
print("membership_row_id unique:", membership["membership_row_id"].is_unique)
print("subscription_days min/max:", membership["subscription_days"].min(), membership["subscription_days"].max())

membership rows: 17876
membership_row_id unique: True
subscription_days min/max: 1 33


## 2-5. 분석 제외 기준

v1 데이터는 이미 팀 합의 전처리가 끝난 기준 데이터입니다.  
여기서 적용하는 것은 raw 전처리가 아니라, 이번 분석용 제외 기준입니다.

현재 적용 기준은 두 가지입니다.

1. 더미 인구통계 케이스 제거  
   `gender == 'N'`, `is_user_verified == 0`, `age == 40`을 동시에 만족하는 행

2. 3주 관측창 구성이 어려운 구독기간 21일 미만 케이스 제거

In [30]:
dummy_demographic_mask = (
    (membership["gender"] == "N")
    & (membership["is_user_verified"] == 0)
    & (membership["age"] == 40)
)

short_subscription_mask = membership["subscription_days"] < 21

keep_mask = ~dummy_demographic_mask & ~short_subscription_mask

filter_summary = pd.DataFrame([
    {"step": "input_membership_v1", "rows": len(membership), "removed": 0, "remaining": len(membership)},
    {
        "step": "remove_dummy_demographic_genderN_verified0_age40",
        "rows": len(membership),
        "removed": int(dummy_demographic_mask.sum()),
        "remaining": int((~dummy_demographic_mask).sum()),
    },
    {
        "step": "remove_subscription_days_lt_21",
        "rows": int((~dummy_demographic_mask).sum()),
        "removed": int((~dummy_demographic_mask & short_subscription_mask).sum()),
        "remaining": int(keep_mask.sum()),
    },
])

membership_preprocessed = membership.loc[keep_mask].copy().reset_index(drop=True)

display(filter_summary)
print("membership_preprocessed rows:", len(membership_preprocessed))

,step,rows,removed,remaining
0,input_membership_v1,17876,0,17876
1,remove_dummy_demographic_genderN_verified0_age40,17876,2639,15237
2,remove_subscription_days_lt_21,15237,315,14922


membership_preprocessed rows: 14922


## 2-6. 분석용 flag 생성

`price == 100`을 `is_100won`으로 정의합니다.  
`is_churn_prevented`는 이번 구독 이벤트의 사후 개입 결과로 단정하지 않고, 과거 해지방어 혜택 수혜 이력이라는 해석 alias를 함께 둡니다.

In [31]:
membership_preprocessed["is_100won"] = (membership_preprocessed["price"] == 100).astype(int)

membership_preprocessed["screen_1_flag"] = (membership_preprocessed["max_screen"] == 1).astype(int)
membership_preprocessed["screen_2_flag"] = (membership_preprocessed["max_screen"] == 2).astype(int)
membership_preprocessed["screen_4_flag"] = (membership_preprocessed["max_screen"] == 4).astype(int)

membership_preprocessed["promo_x_1screen"] = (
    (membership_preprocessed["is_100won"] == 1)
    & (membership_preprocessed["max_screen"] == 1)
).astype(int)

membership_preprocessed["promo_x_2screen"] = (
    (membership_preprocessed["is_100won"] == 1)
    & (membership_preprocessed["max_screen"] == 2)
).astype(int)

membership_preprocessed["promo_x_4screen"] = (
    (membership_preprocessed["is_100won"] == 1)
    & (membership_preprocessed["max_screen"] == 4)
).astype(int)

membership_preprocessed["has_prior_churn_prevention_benefit"] = (
    membership_preprocessed["is_churn_prevented"].astype(int)
)

age_bins = [0, 19, 29, 39, 49, 59, 200]
age_labels = ["10s_or_less", "20s", "30s", "40s", "50s", "60s_plus"]
membership_preprocessed["age_band"] = pd.cut(
    membership_preprocessed["age"],
    bins=age_bins,
    labels=age_labels,
    right=True,
).astype(str)

promo_price_crosstab = pd.crosstab(
    membership_preprocessed["is_promotion"],
    membership_preprocessed["is_100won"],
    rownames=["is_promotion"],
    colnames=["is_100won"],
    dropna=False,
)

display(promo_price_crosstab)

is_100won,0,1
is_promotion,,
0,5939,0
1,0,8983


## 2-7. 100원딜과 요금제 기준 검산

이 표는 이후 전체 스토리의 핵심 기준이므로 보고서용 CSV로 저장합니다.

In [32]:
repurchase_by_100won = (
    membership_preprocessed
    .groupby("is_100won")["is_repurchase"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"count": "n", "mean": "repurchase_rate"})
)

repurchase_by_100won_maxscreen = (
    membership_preprocessed
    .groupby(["is_100won", "max_screen"])["is_repurchase"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"count": "n", "mean": "repurchase_rate"})
)

display(repurchase_by_100won)
display(repurchase_by_100won_maxscreen)

,is_100won,n,repurchase_rate
0,0,5939,0.741034
1,1,8983,0.631749


,is_100won,max_screen,n,repurchase_rate
0,0,1,3794,0.719294
1,0,2,1594,0.784191
2,0,4,551,0.765880
3,1,1,5413,0.658045
4,1,2,1528,0.740183
5,1,4,2042,0.480901


## 2-8. User_Mapping 연결

분석 단위는 개인 단위가 아니라 `membership_row_id` 기준 구독 이벤트 단위입니다.  
일부 `membership_row_id`가 복수 `USER_NUM`과 연결될 수 있으며, 이는 복수 계정 또는 재가입 가능성으로 기록합니다.

In [33]:
membership_with_usernum = membership_preprocessed.merge(
    mapping,
    on="USER_KEY",
    how="left",
    validate="m:m",
)

mapping_join_summary = pd.DataFrame([
    {"metric": "membership_preprocessed_rows", "value": len(membership_preprocessed)},
    {"metric": "membership_with_usernum_rows", "value": len(membership_with_usernum)},
    {"metric": "membership_row_id_nunique_after_join", "value": membership_with_usernum["membership_row_id"].nunique()},
    {"metric": "missing_USER_NUM_rows", "value": int(membership_with_usernum["USER_NUM"].isna().sum())},
    {"metric": "membership_row_id_with_multiple_USER_NUM", "value": int((membership_with_usernum.groupby("membership_row_id")["USER_NUM"].nunique() > 1).sum())},
])

multi_usernum_membership_rows = (
    membership_with_usernum
    .groupby("membership_row_id")["USER_NUM"]
    .nunique()
    .reset_index(name="user_num_count")
    .query("user_num_count > 1")
    .merge(
        membership_preprocessed[["membership_row_id", "USER_KEY", "is_100won", "max_screen", "is_repurchase"]],
        on="membership_row_id",
        how="left",
    )
)

display(mapping_join_summary)
display(multi_usernum_membership_rows.head())

,metric,value
0,membership_preprocessed_rows,14922
1,membership_with_usernum_rows,14955
2,membership_row_id_nunique_after_join,14922
3,missing_USER_NUM_rows,0
4,membership_row_id_with_multiple_USER_NUM,32


,membership_row_id,user_num_count,USER_KEY,is_100won,max_screen,is_repurchase
0,1078,2,e6c1594d93d0d91059c8244b89dbc0414c0054e0aaecea...,0,2,1
1,1505,2,a0218892bc0315bc5a83ac264ab5efd8b90e1ad51b84ea...,1,4,1
2,1889,2,51114f7359992ecf5c70f08e63cbc1b450779dcdb87472...,0,4,0
3,2110,2,07e8934c2a06f8b6e27296413d04e783282f9446ea0c8c...,0,2,1
4,2263,2,63370177497798e110c86c207966d8348c67f2120eda7b...,0,2,0


## 2-9. 고객별 3주 관측창 생성

가입일 `reg_date` 기준 day 0~20만 관측창으로 사용합니다.  
day 21~27은 4주차 대응기간으로 보고 피처 생성에서 제외합니다.

In [34]:
view_base_for_join = view.copy()

membership_for_view = membership_with_usernum[
    [
        "membership_row_id",
        "USER_KEY",
        "USER_NUM",
        "reg_date",
        "end_date",
        "is_repurchase",
        "is_100won",
        "max_screen",
    ]
].dropna(subset=["USER_NUM"]).copy()

obs_view = view_base_for_join.merge(
    membership_for_view,
    on="USER_NUM",
    how="inner",
)

obs_view["watch_rel_day"] = (obs_view["watch_day"] - obs_view["reg_date"]).dt.days

obs_view = obs_view.loc[
    obs_view["watch_rel_day"].between(0, 20, inclusive="both")
].copy()

obs_view["obs_week"] = pd.cut(
    obs_view["watch_rel_day"],
    bins=[-1, 6, 13, 20],
    labels=[1, 2, 3],
).astype(int)

observation_window_summary = pd.DataFrame([
    {"metric": "view_base_rows", "value": len(view_base_for_join)},
    {"metric": "obs_view_rows_day0_20", "value": len(obs_view)},
    {"metric": "obs_membership_row_id_nunique", "value": obs_view["membership_row_id"].nunique()},
    {"metric": "obs_user_num_nunique", "value": obs_view["USER_NUM"].nunique()},
    {"metric": "obs_movie_num_nunique", "value": obs_view["MOVIE_NUM"].nunique()},
    {"metric": "watch_rel_day_min", "value": int(obs_view["watch_rel_day"].min()) if len(obs_view) else np.nan},
    {"metric": "watch_rel_day_max", "value": int(obs_view["watch_rel_day"].max()) if len(obs_view) else np.nan},
])

display(observation_window_summary)

,metric,value
0,view_base_rows,106205
1,obs_view_rows_day0_20,85066
2,obs_membership_row_id_nunique,12302
3,obs_user_num_nunique,12303
4,obs_movie_num_nunique,4765
5,watch_rel_day_min,0
6,watch_rel_day_max,20


## 2-10. 시청이력 존재 여부 요약

시청이력 없는 구독 이벤트는 삭제하지 않고 이후 단계에서 `no_watch_obs_flag`로 처리합니다.

In [35]:
watch_presence = membership_preprocessed[["membership_row_id", "is_100won", "max_screen", "is_repurchase"]].copy()
watched_ids = set(obs_view["membership_row_id"].unique())
watch_presence["has_watch_obs"] = watch_presence["membership_row_id"].isin(watched_ids).astype(int)
watch_presence["no_watch_obs_flag"] = (watch_presence["has_watch_obs"] == 0).astype(int)

watch_presence_summary = (
    watch_presence
    .groupby("has_watch_obs")["is_repurchase"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"count": "n", "mean": "repurchase_rate"})
)

display(watch_presence_summary)

,has_watch_obs,n,repurchase_rate
0,0,2620,0.676718
1,1,12302,0.674931


In [36]:
print("view rows:", len(view))
print("membership_with_usernum rows:", len(membership_with_usernum))

print("view USER_NUM dtype:", view["USER_NUM"].dtype)
print("membership USER_NUM dtype:", membership_with_usernum["USER_NUM"].dtype)

tmp_join = view.merge(
    membership_with_usernum[
        ["membership_row_id", "USER_NUM", "reg_date", "end_date"]
    ].dropna(subset=["USER_NUM"]),
    on="USER_NUM",
    how="inner",
)

print("tmp_join rows:", len(tmp_join))

tmp_join["watch_rel_day"] = (tmp_join["watch_day"] - tmp_join["reg_date"]).dt.days

print("watch_day min/max:", tmp_join["watch_day"].min(), tmp_join["watch_day"].max())
print("reg_date min/max:", tmp_join["reg_date"].min(), tmp_join["reg_date"].max())
print("watch_rel_day min/max:", tmp_join["watch_rel_day"].min(), tmp_join["watch_rel_day"].max())

print("watch_rel_day 0~20 rows:", tmp_join["watch_rel_day"].between(0, 20, inclusive="both").sum())
print("watch_rel_day -20~0 rows:", tmp_join["watch_rel_day"].between(-20, 0, inclusive="both").sum())

display(
    tmp_join["watch_rel_day"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)

view rows: 106205
membership_with_usernum rows: 14955
view USER_NUM dtype: int64
membership USER_NUM dtype: int64
tmp_join rows: 89100
watch_day min/max: 2021-03-01 00:00:00 2021-04-05 00:00:00
reg_date min/max: 2021-03-01 00:00:00 2021-03-15 00:00:00
watch_rel_day min/max: -12 31
watch_rel_day 0~20 rows: 85066
watch_rel_day -20~0 rows: 4705


count    89100.000000
mean        10.435376
std          6.380211
min        -12.000000
1%           0.000000
5%           0.000000
25%          5.000000
50%         10.000000
75%         16.000000
95%         20.000000
99%         21.000000
max         31.000000
Name: watch_rel_day, dtype: float64

## 2-11. 산출물 저장

02번에서 저장하는 산출물은 다음으로 제한합니다.

데이터 산출물:
- `membership_preprocessed.csv`
- `membership_with_usernum.csv`
- `view_history_observation_window.csv`
- `preprocessing_summary.json`

보고서용 검산표:
- `02_filter_summary.csv`
- `02_repurchase_by_100won.csv`
- `02_repurchase_by_100won_maxscreen.csv`
- `02_user_mapping_summary.csv`
- `02_multi_usernum_membership_rows.csv`
- `02_observation_window_summary.csv`
- `02_watch_presence_summary.csv`

In [37]:
membership_preprocessed.to_csv(
    OUTPUT_DATA_DIR / "membership_preprocessed.csv",
    index=False,
    encoding="utf-8-sig",
)

membership_with_usernum.to_csv(
    OUTPUT_DATA_DIR / "membership_with_usernum.csv",
    index=False,
    encoding="utf-8-sig",
)

obs_view.to_csv(
    OUTPUT_DATA_DIR / "view_history_observation_window.csv",
    index=False,
    encoding="utf-8-sig",
)

filter_summary.to_csv(OUTPUT_TABLE_DIR / "02_filter_summary.csv", index=False, encoding="utf-8-sig")
repurchase_by_100won.to_csv(OUTPUT_TABLE_DIR / "02_repurchase_by_100won.csv", index=False, encoding="utf-8-sig")
repurchase_by_100won_maxscreen.to_csv(OUTPUT_TABLE_DIR / "02_repurchase_by_100won_maxscreen.csv", index=False, encoding="utf-8-sig")
mapping_join_summary.to_csv(OUTPUT_TABLE_DIR / "02_user_mapping_summary.csv", index=False, encoding="utf-8-sig")
multi_usernum_membership_rows.to_csv(OUTPUT_TABLE_DIR / "02_multi_usernum_membership_rows.csv", index=False, encoding="utf-8-sig")
observation_window_summary.to_csv(OUTPUT_TABLE_DIR / "02_observation_window_summary.csv", index=False, encoding="utf-8-sig")
watch_presence_summary.to_csv(OUTPUT_TABLE_DIR / "02_watch_presence_summary.csv", index=False, encoding="utf-8-sig")

preprocessing_summary = {
    "notebook_id": NOTEBOOK_ID,
    "project_root": str(PROJECT_ROOT),
    "v1_data_dir": str(V1_DATA_DIR),
    "output_data_dir": str(OUTPUT_DATA_DIR),
    "membership_base_rows": int(len(membership)),
    "membership_preprocessed_rows": int(len(membership_preprocessed)),
    "removed_dummy_demographic_rows": int(dummy_demographic_mask.sum()),
    "removed_subscription_days_lt_21_rows": int((~dummy_demographic_mask & short_subscription_mask).sum()),
    "preprocessed_100won_rows": int((membership_preprocessed["is_100won"] == 1).sum()),
    "preprocessed_non_100won_rows": int((membership_preprocessed["is_100won"] == 0).sum()),
    "membership_with_usernum_rows": int(len(membership_with_usernum)),
    "membership_row_id_with_multiple_USER_NUM": int((membership_with_usernum.groupby("membership_row_id")["USER_NUM"].nunique() > 1).sum()),
    "observation_window_rows_day0_20": int(len(obs_view)),
    "obs_membership_row_id_nunique": int(obs_view["membership_row_id"].nunique()),
    "watch_presence_rows_without_watch": int((watch_presence["has_watch_obs"] == 0).sum()),
}

with open(OUTPUT_DATA_DIR / "preprocessing_summary.json", "w", encoding="utf-8") as f:
    json.dump(preprocessing_summary, f, ensure_ascii=False, indent=2)

print("Saved data outputs to:", OUTPUT_DATA_DIR)
print("Saved report tables to:", OUTPUT_TABLE_DIR)

Saved data outputs to: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\02_preprocessing_policy
Saved report tables to: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables\02_preprocessing_policy


## 2-12. 최종 검산

아래 조건이 만족되면 02번은 통과로 본다.

- `PROJECT_ROOT`가 저장소 루트로 잡힌다.
- `DATA_ROOT`가 저장소 루트의 `_data`로 잡힌다.
- `REPORTS_DIR`가 `park.ingyeom/reports`로 잡힌다.
- 02번 report CSV는 의도한 7개만 남는다.
- `membership_preprocessed`는 14,922행 기준이 된다.
- 100원딜 고객 수는 8,983명 기준이 된다.
- 관측창은 `watch_rel_day` 0~20만 포함한다.

In [38]:
expected_report_files = {
    "02_filter_summary.csv",
    "02_repurchase_by_100won.csv",
    "02_repurchase_by_100won_maxscreen.csv",
    "02_user_mapping_summary.csv",
    "02_multi_usernum_membership_rows.csv",
    "02_observation_window_summary.csv",
    "02_watch_presence_summary.csv",
}

actual_02_report_files = {p.name for p in OUTPUT_TABLE_DIR.glob("02_*.csv")}

final_checks = pd.DataFrame([
    {"check": "project_root_is_repo_root", "value": str(PROJECT_ROOT), "pass": (PROJECT_ROOT / ".git").exists()},
    {"check": "data_root_is_repo_data", "value": str(DATA_ROOT), "pass": DATA_ROOT.exists()},
    {"check": "reports_dir_is_park_reports", "value": str(REPORTS_DIR), "pass": REPORTS_DIR == WORK_ROOT / "reports"},
    {"check": "only_expected_02_report_csvs", "value": sorted(actual_02_report_files), "pass": actual_02_report_files == expected_report_files},
    {"check": "membership_preprocessed_rows", "value": len(membership_preprocessed), "pass": len(membership_preprocessed) == 14922},
    {"check": "preprocessed_100won_rows", "value": int((membership_preprocessed["is_100won"] == 1).sum()), "pass": int((membership_preprocessed["is_100won"] == 1).sum()) == 8983},
    {"check": "obs_watch_rel_day_min", "value": int(obs_view["watch_rel_day"].min()) if len(obs_view) else None, "pass": len(obs_view) > 0 and obs_view["watch_rel_day"].min() >= 0},
    {"check": "obs_watch_rel_day_max", "value": int(obs_view["watch_rel_day"].max()) if len(obs_view) else None, "pass": len(obs_view) > 0 and obs_view["watch_rel_day"].max() <= 20},
    {"check": "output_data_dir_under_reports_data_02", "value": str(OUTPUT_DATA_DIR), "pass": OUTPUT_DATA_DIR == REPORTS_DIR / "data" / NOTEBOOK_ID},
	{"check": "output_table_dir_under_reports_tables_02", "value": str(OUTPUT_TABLE_DIR), "pass": OUTPUT_TABLE_DIR == REPORTS_DIR / "tables" / NOTEBOOK_ID},
])

display(final_checks)

if not final_checks["pass"].all():
    failed = final_checks.loc[~final_checks["pass"]]
    raise AssertionError(f"02번 최종 검산 실패:\n{failed}")

print("02_preprocessing_policy passed final checks.")

,check,value,pass
0,project_root_is_repo_root,c:\Code\ott-churn-prediction,True
1,data_root_is_repo_data,c:\Code\ott-churn-prediction\_data,True
2,reports_dir_is_park_reports,c:\Code\ott-churn-prediction\park.ingyeom\reports,True
3,only_expected_02_report_csvs,"[02_filter_summary.csv, 02_multi_usernum_membe...",True
4,membership_preprocessed_rows,14922,True
5,preprocessed_100won_rows,8983,True
6,obs_watch_rel_day_min,0,True
7,obs_watch_rel_day_max,20,True
8,output_data_dir_under_reports_data_02,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
9,output_table_dir_under_reports_tables_02,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True


02_preprocessing_policy passed final checks.
